<a href="https://colab.research.google.com/github/XiaRui1996/GPLAR/blob/master/VAE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import tensorflow as tf
import numpy as np
import re
import copy

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import random
from random import sample
import pandas as pd
import sklearn.preprocessing as preprocessing

from scipy.stats import bernoulli
import argparse
import os

In [8]:
import argparse


In [2]:
class PNPlusEncoder(tf.keras.layers.Layer):
    """
    PN+ style partial encoder:
    per-feature encode -> masked sum pooling -> MLP -> (mu, logvar)
    """
    def __init__(self, obs_dim: int, K: int, latent_dim: int, id_dim: int = 10, name="pnplus_encoder"):
        super().__init__(name=name)
        self.obs_dim = obs_dim
        self.K = K
        self.latent_dim = latent_dim
        self.id_dim = id_dim

        # Learnable per-feature "ID" embedding F and bias b (like your TF1 F,b variables)
        self.F = self.add_weight(
            name="F", shape=(obs_dim, id_dim),
            initializer=tf.keras.initializers.GlorotUniform(), trainable=True
        )
        self.b = self.add_weight(
            name="b", shape=(obs_dim, 1),
            initializer=tf.keras.initializers.GlorotUniform(), trainable=True
        )

        # Per-feature projection to K (applied on last dim of [B,D,?])
        self.fc_feat = tf.keras.layers.Dense(K, activation=None, name="fc_feat")

        # Post-pooling MLP
        self.fc1 = tf.keras.layers.Dense(500, activation="relu", name="fc1")
        self.fc2 = tf.keras.layers.Dense(200, activation="relu", name="fc2")
        self.fc_out = tf.keras.layers.Dense(2 * latent_dim, activation=None, name="fc_out")

    def call(self, x, mask, training=False):
        """
        x:    [B, D]
        mask: [B, D] (1 observed, 0 missing)
        """
        x = tf.convert_to_tensor(x, dtype=tf.float32)
        mask = tf.convert_to_tensor(mask, dtype=tf.float32)

        B = tf.shape(x)[0]
        D = self.obs_dim

        # [B,D,1]
        x_ = tf.reshape(x, [B, D, 1])

        # Broadcast F,b to batch:
        # F: [D,id_dim] -> [B,D,id_dim]
        F = tf.broadcast_to(self.F[None, :, :], [B, D, self.id_dim])
        # b: [D,1] -> [B,D,1]
        b = tf.broadcast_to(self.b[None, :, :], [B, D, 1])

        # x_aug = concat([x, x*F, b]) along last dim
        # x*F: [B,D,1] * [B,D,id_dim] -> [B,D,id_dim]
        xF = x_ * F
        x_aug = tf.concat([x_, xF, b], axis=-1)   # [B,D, 1+id_dim+1]

        # Per-feature encode to K
        h = self.fc_feat(x_aug)                  # [B,D,K]

        # Masked sum pooling over features
        mask_h = tf.expand_dims(mask, axis=-1)   # [B,D,1]
        h = tf.reduce_sum(h * mask_h, axis=1)    # [B,K]
        h = tf.nn.relu(h)

        # MLP -> (mu, logvar)
        h = self.fc1(h)
        h = self.fc2(h)
        out = self.fc_out(h)                     # [B,2*latent_dim]

        mu = out[:, :self.latent_dim]
        logvar = out[:, self.latent_dim:]
        return mu, logvar


class PNPlusVAE(tf.keras.Model):
    """
    TF2/Keras version of your PN_Plus_VAE.
    decoder: a keras Layer/Model that maps z -> decoded_mean/prob with shape [B, obs_dim]
    """
    def __init__(
        self,
        obs_dim: int,
        decoder,
        K: int = 20,
        latent_dim: int = 10,
        obs_distrib: str = "Gaussian",
        obs_std: float = 0.1 * np.sqrt(2),
        learning_rate: float = 1e-3,
        M: int = 5,
        name="pnplus_vae",
    ):
        super().__init__(name=name)
        self.obs_dim = obs_dim
        self.K = K
        self.latent_dim = latent_dim
        self.obs_distrib = obs_distrib
        self.obs_std = tf.constant(obs_std, dtype=tf.float32)
        self.M = M

        self.encoder = PNPlusEncoder(obs_dim=obs_dim, K=K, latent_dim=latent_dim)
        self.decoder = decoder  # must be callable: decoded = decoder(z) -> [B,obs_dim]

        self.optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    @staticmethod
    def _kl_diag_normal_stdnormal(mu, logvar):
        # KL(q(z|x) || N(0,I)) for diagonal Gaussian
        # 0.5 * sum(mu^2 + exp(logvar) - 1 - logvar)
        return 0.5 * tf.reduce_sum(tf.square(mu) + tf.exp(logvar) - 1.0 - logvar, axis=-1)  # [B]

    def _recon_nll(self, x, decoded, mask):
        """
        Negative log-likelihood term (masked).
        Returns per-sample recon loss: shape [B]
        """
        x = tf.convert_to_tensor(x, tf.float32)
        decoded = tf.convert_to_tensor(decoded, tf.float32)
        mask = tf.convert_to_tensor(mask, tf.float32)

        if self.obs_distrib.lower() == "bernoulli":
            eps = tf.constant(1e-8, tf.float32)
            # decoded assumed to be probabilities in (0,1)
            nll_entry = -(x * tf.math.log(decoded + eps) + (1. - x) * tf.math.log(1. - decoded + eps))
            nll_entry = nll_entry * mask
            return tf.reduce_sum(nll_entry, axis=-1)  # [B]
        else:
            # Gaussian with fixed std (drop constants)
            # NOTE: correctly mask per-entry so missing dims don't contribute
            std = self.obs_std
            nll_entry = 0.5 * tf.square(x - decoded) / (std ** 2) + tf.math.log(std)
            nll_entry = nll_entry * mask

            # If you want to replicate your TF1 behavior exactly (not recommended),
            # you would NOT multiply by mask here, and instead feed x*mask, decoded*mask.
            return tf.reduce_sum(nll_entry, axis=-1)  # [B]

    def call(self, inputs, training=False):
        """
        inputs: (x, mask)
        returns: decoded, mu, logvar, z
        """
        x, mask = inputs
        mu, logvar = self.encoder(x, mask, training=training)
        eps = tf.random.normal(shape=tf.shape(mu), dtype=tf.float32)
        z = mu + tf.exp(0.5 * logvar) * eps
        decoded = self.decoder(z, training=training)
        return decoded, mu, logvar, z

    def compute_losses(self, x, mask):
        decoded, mu, logvar, _ = self((x, mask), training=True)
        kl = self._kl_diag_normal_stdnormal(mu, logvar)     # [B]
        recon = self._recon_nll(x, decoded, mask)           # [B]

        loss_per_instance = tf.reduce_mean(kl + recon)

        # like your _loss_print: averaged per observed feature count (for tracking)
        denom = tf.reduce_sum(mask) + 1e-8
        loss_print = tf.reduce_sum(kl + recon) / denom
        return loss_per_instance, loss_print, tf.reduce_sum(kl), tf.reduce_sum(recon)

    @tf.function
    def train_step_tf(self, x, mask):
        with tf.GradientTape() as tape:
            loss, loss_print, kl_sum, recon_sum = self.compute_losses(x, mask)
        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        return loss_print, kl_sum, recon_sum

    # --- API that mirrors your TF1 code ---

    def update(self, x, mask):
        loss_print, _, _ = self.train_step_tf(x, mask)
        return float(loss_print.numpy())

    def full_batch_loss(self, x, mask):
        decoded, mu, logvar, _ = self((x, mask), training=False)
        kl = tf.reduce_sum(self._kl_diag_normal_stdnormal(mu, logvar))
        recon = tf.reduce_sum(self._recon_nll(x, decoded, mask))
        denom = tf.reduce_sum(mask) + 1e-8
        loss_print = (kl + recon) / denom
        return float(loss_print.numpy()), float(kl.numpy()), float(recon.numpy())

    def impute_once(self, x, mask):
        decoded, _, _, _ = self((x, mask), training=False)
        return decoded.numpy()

    def predictive_loss(self, x, mask, eval="rmse", M=None):
        """
        Assumes last column is the target variable (like your TF1 code).
        """
        if M is None:
            M = self.M

        x = np.asarray(x, dtype=np.float32)
        mask = np.asarray(mask, dtype=np.float32)

        target = x[:, -1]
        preds = np.zeros((x.shape[0], M), dtype=np.float32)

        for m in range(M):
            decoded = self.impute_once(x, mask)
            preds[:, m] = decoded[:, -1]

        pred_mean = preds.mean(axis=1)
        uncertainty = preds.std(axis=1)

        if eval == "rmse":
            mse = (target - pred_mean) ** 2
            return mse, uncertainty
        else:
            # Negative log-likelihood of target only
            if self.obs_distrib.lower() == "bernoulli":
                eps = 1e-8
                p = np.clip(pred_mean, eps, 1 - eps)
                nll = -(target * np.log(p) + (1 - target) * np.log(1 - p))
                return nll, uncertainty
            else:
                std = float(self.obs_std.numpy())
                # Gaussian NLL (include constants if you want; here match your style loosely)
                nll = 0.5 * ((target - pred_mean) ** 2) / (std ** 2) + np.log(std)
                return nll, uncertainty

    def impute_losses(self, x, mask_obs, mask_target):


        x = np.asarray(x, dtype=np.float32)
        mask_obs = np.asarray(mask_obs, dtype=np.float32)
        mask_target = np.asarray(mask_target, dtype=np.float32)

        denom = np.sum(mask_target) + 1e-8

        SE = 0.0
        RMSE = 0.0

        for _ in range(self.M):
            decoded = self.impute_once(x, mask_obs)   # [N,D] numpy

            target = x * mask_target
            output = decoded * mask_target

            se = np.sum((target - output) ** 2)
            SE += se
            RMSE += np.sqrt(se / denom)

        SE /= self.M
        RMSE /= self.M
        return SE, RMSE

    # ---- acquisition helpers (chaini_I / chaini_II) ----

    def _posterior_params_np(self, x, mask):
        x = np.asarray(x, dtype=np.float32)
        mask = np.asarray(mask, dtype=np.float32)
        mu, logvar = self.encoder(x, mask, training=False)
        mu = mu.numpy()
        var = np.exp(logvar.numpy())  # variance
        return mu, var

    @staticmethod
    def _kl_diag_gaussians(mu_p, var_p, mu_q, var_q):
        """
        KL( N(mu_p,var_p) || N(mu_q,var_q) ) for diagonal Gaussians.
        All inputs: [B, latent_dim]
        """
        # 0.5 * sum( log(var_q/var_p) + (var_p + (mu_p-mu_q)^2)/var_q - 1 )
        return 0.5 * np.sum(
            np.log(var_q / (var_p + 1e-12) + 1e-12) +
            (var_p + (mu_p - mu_q) ** 2) / (var_q + 1e-12) - 1.0,
            axis=1
        )

    def chaini_I(self, x, mask, i):
        """
        First term: KL(q(z|x_O, x_i) || q(z|x_O))
        x: [N,D] or [1,D]
        mask: same shape
        """
        temp_mask = copy.deepcopy(mask)
        mu, var = self._posterior_params_np(x, temp_mask)

        temp_mask[:, i] = 1.0
        mu_i, var_i = self._posterior_params_np(x, temp_mask)

        kl = self._kl_diag_gaussians(mu_i, var_i, mu, var)
        return kl

    def chaini_II(self, x, mask, i):
        """
        Second term: KL(q(z|x_O, x_i, x_target) || q(z|x_O, x_target))
        Assumes last column is target -> force mask[:, -1] = 1
        """
        temp_mask = copy.deepcopy(mask)
        temp_mask[:, -1] = 1.0
        mu, var = self._posterior_params_np(x, temp_mask)

        temp_mask[:, i] = 1.0
        mu_i, var_i = self._posterior_params_np(x, temp_mask)

        kl = self._kl_diag_gaussians(mu_i, var_i, mu, var)
        return kl

    # ---- saving/loading ----

    def save_encoder(self, path):
        # encoder weights only
        self.encoder.save_weights(path)

    def save_decoder(self, path):
        self.decoder.save_weights(path)

    def load_encoder(self, path):
        self.encoder.load_weights(path)

    def load_decoder(self, path):
        self.decoder.load_weights(path)


# ---------- utilities matching your completion / reward code ----------

def completion(x, mask, M, vae: PNPlusVAE):
    """
    Generate M imputations conditioned on observed x_O (via sampling in VAE).
    Returns im: [M, N, D]
    """
    x = np.asarray(x, dtype=np.float32)
    mask = np.asarray(mask, dtype=np.float32)

    im = np.zeros((M, x.shape[0], x.shape[1]), dtype=np.float32)
    for m in range(M):
        im[m] = vae.impute_once(x, mask)
    return im


def R_lindley_chain(i, x, mask, M, vae: PNPlusVAE, im, loc):
    """
    Approx reward: E[ KL_I - KL_II ]
    loc: index of the instance you are acquiring for
    """
    im_i = im[:, :, i]
    im_target = im[:, :, -1]
    approx = 0.0

    temp_x = copy.deepcopy(x)
    for m in range(M):
        temp_x[loc, i] = im_i[m, loc]
        KL_I = vae.chaini_I(temp_x[loc:loc+1, :], mask[loc:loc+1, :], i)[0]

        temp_x[loc, -1] = im_target[m, loc]
        KL_II = vae.chaini_II(temp_x[loc:loc+1, :], mask[loc:loc+1, :], i)[0]

        approx += (KL_I - KL_II)

    return approx / M

In [3]:
class FCUCIDecoder(tf.keras.Model):
    def __init__(self, obs_dim, final_activation="sigmoid", name="fc_uci_decoder"):
        super().__init__(name=name)
        self.fc01 = tf.keras.layers.Dense(50, activation="relu", name="fc-01")
        self.fc02 = tf.keras.layers.Dense(100, activation="relu", name="fc-02")
        self.fc_final = tf.keras.layers.Dense(
            obs_dim,
            activation=final_activation,   # "sigmoid" for Bernoulli; None for Gaussian mean
            name="fc-final"
        )

    def call(self, z, training=False):
        x = self.fc01(z)
        x = self.fc02(x)
        x = self.fc_final(x)
        return x  # shape [B, obs_dim]

class FCUCIEncoder(tf.keras.Model):
    """
    TF2 replacement for fc_uci_encoder
    Outputs 2*latent_dim (mu and logvar concatenated).
    """
    def __init__(self, latent_dim, final_activation=None, name="fc_uci_encoder"):
        super().__init__(name=name)
        self.fc01 = tf.keras.layers.Dense(100, activation="relu", name="fc-01")
        self.fc02 = tf.keras.layers.Dense(50, activation="relu", name="fc-02")
        self.fc_final = tf.keras.layers.Dense(
            2 * latent_dim,
            activation=final_activation,   # usually None
            name="fc-final"
        )

    def call(self, x, training=False):
        e = self.fc01(x)
        e = self.fc02(e)
        e = self.fc_final(e)
        return e

class PNPFCUCIEncoder(tf.keras.Model):
    """
    TF2 replacement for PNP_fc_uci_encoder
    Outputs K-dim feature embedding.
    """
    def __init__(self, K, name="pnp_fc_uci_encoder"):
        super().__init__(name=name)
        self.fc01 = tf.keras.layers.Dense(100, activation="relu", name="fc-01")
        self.fc02 = tf.keras.layers.Dense(50, activation="relu", name="fc-02")
        self.fc_final = tf.keras.layers.Dense(K, activation=None, name="fc-final")

    def call(self, x, training=False):
        e = self.fc01(x)
        e = self.fc02(e)
        e = self.fc_final(e)
        return e

In [4]:
!pip install ucimlrepo
from ucimlrepo import fetch_ucirepo

# fetch dataset
wine_quality = fetch_ucirepo(id=186)

# data (as pandas dataframes)
X = wine_quality.data.features
y = wine_quality.data.targets

# metadata
print(wine_quality.metadata)

# variable information
print(wine_quality.variables)


{'uci_id': 186, 'name': 'Wine Quality', 'repository_url': 'https://archive.ics.uci.edu/dataset/186/wine+quality', 'data_url': 'https://archive.ics.uci.edu/static/public/186/data.csv', 'abstract': 'Two datasets are included, related to red and white vinho verde wine samples, from the north of Portugal. The goal is to model wine quality based on physicochemical tests (see [Cortez et al., 2009], http://www3.dsi.uminho.pt/pcortez/wine/).', 'area': 'Business', 'tasks': ['Classification', 'Regression'], 'characteristics': ['Multivariate'], 'num_instances': 4898, 'num_features': 11, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['quality'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2009, 'last_updated': 'Wed Nov 15 2023', 'dataset_doi': '10.24432/C56S3T', 'creators': ['Paulo Cortez', 'A. Cerdeira', 'F. Almeida', 'T. Matos', 'J. Reis'], 'intro_paper': {'ID': 252, 'type': 'NATIVE', 'title': 'Modeling wine preferences

In [5]:
## training vae

latent_dim = 10
K = 20
batch_size = 100
p = 0.7
M = 50

rs = 42
random.seed(rs)
np.random.seed(rs)
tf.random.set_seed(rs)

# ----------------------------
# build / save / load
# ----------------------------
def build_pvae(obs_dim, latent_dim, K, obs_distrib="Gaussian",
              obs_std=0.1*np.sqrt(2), M=50, learning_rate=1e-3):
    final_act = None if obs_distrib.lower() == "gaussian" else "sigmoid"
    decoder = FCUCIDecoder(obs_dim=obs_dim, final_activation=final_act)

    vae = PNPlusVAE(
        obs_dim=obs_dim,
        decoder=decoder,
        K=K,
        latent_dim=latent_dim,
        obs_distrib=obs_distrib,
        obs_std=obs_std,
        learning_rate=learning_rate,
        M=M,
    )

    # 先 build 一次（很关键）
    dummy_x = np.zeros((1, obs_dim), dtype=np.float32)
    dummy_m = np.ones((1, obs_dim), dtype=np.float32)
    _ = vae((dummy_x, dummy_m), training=False)
    return vae


# ----------------------------
# save / load (save whole VAE)
# ----------------------------
def default_weights_path(save_dir="./pvae_ckpt", prefix="wine"):
    os.makedirs(save_dir, exist_ok=True)
    return os.path.join(save_dir, f"{prefix}.pvae.weights.h5")


def save_pvae(vae, save_dir="./pvae_ckpt", prefix="wine"):
    path = default_weights_path(save_dir, prefix)
    vae.save_weights(path)
    print("Saved PVAE weights:", path)
    return path


def load_pvae(obs_dim, latent_dim, K, save_dir="./pvae_ckpt", prefix="wine",
              obs_distrib="Gaussian", obs_std=0.1*np.sqrt(2), M=50, learning_rate=1e-3):
    path = default_weights_path(save_dir, prefix)
    if not os.path.exists(path):
        raise FileNotFoundError(f"weights not found: {path}")

    vae = build_pvae(obs_dim, latent_dim, K,
                     obs_distrib=obs_distrib, obs_std=obs_std, M=M, learning_rate=learning_rate)
    vae.load_weights(path)
    print("Loaded PVAE weights:", path)
    return vae


# ----------------------------
# imputation helper
# ----------------------------
def get_imputation_tf2(vae, x, mask_obs, M=50, clip01=False):
    x = np.asarray(x, dtype=np.float32)
    mask_obs = np.asarray(mask_obs, dtype=np.float32)

    samples = []
    for _ in range(M):
        decoded = vae.impute_once(x, mask_obs)  # [N,D]
        samples.append(decoded)
    mean_pred = np.mean(np.stack(samples, axis=0), axis=0)

    if clip01:
        mean_pred = np.clip(mean_pred, 0.0, 1.0)

    x_fill = x * mask_obs + mean_pred * (1.0 - mask_obs)
    return x_fill


# ----------------------------
# train / test / impute (use trained vae)
# ----------------------------
def train_p_vae(Data_train, mask_train, epochs, latent_dim, batch_size, p, K, iteration,
                obs_distrib="Gaussian", obs_std=0.1*np.sqrt(2), M=50, learning_rate=1e-3,
                save_dir="./pvae_ckpt", prefix="wine", save_weights=True):

    Data_train = np.asarray(Data_train, dtype=np.float32)
    mask_train = np.asarray(mask_train, dtype=np.float32)

    obs_dim = Data_train.shape[1]
    n_train = Data_train.shape[0]
    idx_all = np.arange(n_train)

    vae = build_pvae(obs_dim, latent_dim, K,
                     obs_distrib=obs_distrib, obs_std=obs_std, M=M, learning_rate=learning_rate)

    n_it = int(np.ceil(n_train / float(batch_size))) if iteration == -1 else int(iteration)

    for epoch in range(epochs):
        training_loss_full = 0.0

        for it in range(n_it):
            if iteration == -1:
                start = it * batch_size
                end = min(start + batch_size, n_train)
                batch_idx = idx_all[start:end]
            else:
                batch_idx = sample(range(n_train), min(batch_size, n_train))

            x = Data_train[batch_idx]
            mask_train_batch = mask_train[batch_idx]

            B = x.shape[0]
            DROPOUT_TRAIN = np.minimum(np.random.rand(B, obs_dim), p).astype(np.float32)

            while True:
                mask_drop = bernoulli.rvs(1 - DROPOUT_TRAIN).astype(np.float32)
                if np.sum(mask_drop > 0) > 0:
                    break

            mask_eff = mask_drop * mask_train_batch

            _ = vae.update(x, mask_eff)
            loss_full, _, _ = vae.full_batch_loss(x, mask_eff)
            training_loss_full += loss_full

        training_loss_full /= n_it
        print(f"Epoch: {epoch}\tnegative training ELBO per observed feature: {training_loss_full:.2f}")

    if save_weights:
        save_pvae(vae, save_dir=save_dir, prefix=prefix)

    return vae


def test_p_vae_marginal_elbo(vae, Data_test, mask_test):
    Data_test = np.asarray(Data_test, dtype=np.float32)
    mask_test = np.asarray(mask_test, dtype=np.float32)
    loss, kl, recon = vae.full_batch_loss(Data_test, mask_test)
    print(f"test negative ELBO per feature: {loss:.2f} (KL sum={kl:.2f}, Recon sum={recon:.2f})")
    return loss


def impute_p_vae(vae, Data_ground_truth, mask_obs, mask_target, M=50, clip01=False):
    Data_ground_truth = np.asarray(Data_ground_truth, dtype=np.float32)
    mask_obs = np.asarray(mask_obs, dtype=np.float32)
    mask_target = np.asarray(mask_target, dtype=np.float32)

    SE, RMSE = vae.impute_losses(Data_ground_truth, mask_obs, mask_target, M=M)
    X_fill = get_imputation_tf2(vae, Data_ground_truth, mask_obs, M=M, clip01=clip01)

    print(f"impute RMSE: {RMSE:.6f}")
    return RMSE, X_fill

In [6]:
from sklearn.model_selection import train_test_split
X_df = wine_quality.data.features
y_df = wine_quality.data.targets  # 通常列名是 quality

# 拼接：保证 target 在最后一列（EDDI 假设 x_phi 是最后一列）
Data_df = pd.concat([X_df, y_df], axis=1).dropna().reset_index(drop=True)

# min-max 到 [0,1]（按列）
mins = Data_df.min(axis=0)
maxs = Data_df.max(axis=0)
den = (maxs - mins).replace(0, 1.0)
Data = ((Data_df - mins) / den).to_numpy(dtype=np.float32)

print("Data shape:", Data.shape, "target col:", Data_df.columns[-1])

Data shape: (6497, 12) target col: quality


In [7]:
Data_train, Data_test = train_test_split(Data, test_size=0.1, random_state=rs, shuffle=True)

mask_train = np.ones_like(Data_train, dtype=np.float32)
mask_test  = np.ones_like(Data_test,  dtype=np.float32)

print("Train:", Data_train.shape, "Test:", Data_test.shape)

Train: (5847, 12) Test: (650, 12)


In [18]:
vae = train_p_vae(
    Data_train, mask_train,
    epochs=100, latent_dim=10, batch_size=100, p=0.7, K=20, iteration=-1,
    M=50,
    save_dir="./pvae_ckpt",
    prefix="wine",
    save_weights=True
)

Epoch: 0	negative training ELBO per observed feature: -1.13
Epoch: 1	negative training ELBO per observed feature: -1.49
Epoch: 2	negative training ELBO per observed feature: -1.56
Epoch: 3	negative training ELBO per observed feature: -1.60
Epoch: 4	negative training ELBO per observed feature: -1.62
Epoch: 5	negative training ELBO per observed feature: -1.63
Epoch: 6	negative training ELBO per observed feature: -1.64
Epoch: 7	negative training ELBO per observed feature: -1.65
Epoch: 8	negative training ELBO per observed feature: -1.65
Epoch: 9	negative training ELBO per observed feature: -1.65
Epoch: 10	negative training ELBO per observed feature: -1.65
Epoch: 11	negative training ELBO per observed feature: -1.66
Epoch: 12	negative training ELBO per observed feature: -1.66
Epoch: 13	negative training ELBO per observed feature: -1.66
Epoch: 14	negative training ELBO per observed feature: -1.66
Epoch: 15	negative training ELBO per observed feature: -1.66
Epoch: 16	negative training ELBO p

In [19]:
miss_rate = 0.3
rng = np.random.default_rng(rs)

mask_obs = (rng.random(size=Data_test.shape) > miss_rate).astype(np.float32)
Data_observed = Data_test * mask_obs   # 或者缺失处用列均值填更稳

mask_target = (1.0 - mask_obs)  # Data_test 完整的话就这样即可

SE, RMSE = vae.impute_losses(Data_test, mask_obs, mask_target)
print("Imputation RMSE:", RMSE)

Imputation RMSE: 0.108956784


In [21]:
miss_rate = 0.3
rng = np.random.default_rng(rs)

mask_obs = (rng.random(size=Data_test.shape) > miss_rate).astype(np.float32)

# 强制 target 不可见：评估“补 target”
mask_obs[:, -1] = 0.0

# 只在 target 上评估
mask_target = np.zeros_like(mask_obs, dtype=np.float32)
mask_target[:, -1] = 1.0

SE, RMSE = vae.impute_losses(Data_test, mask_obs, mask_target)
print("Target-only Imputation RMSE:", RMSE)

Target-only Imputation RMSE: 0.14395317


In [7]:
def p_vae_active_learning(
    Data_train, mask_train,
    Data_test, mask_test,
    epochs, latent_dim, batch_size,
    p, K, M, eval, Repeat,
    estimation_method=0,
    encoder_weights_path=None,
    decoder_weights_path=None,
    output_dir=".",
    obs_distrib="Gaussian",
    obs_std=0.1 * np.sqrt(2),
):


    Data_train = np.asarray(Data_train, dtype=np.float32)
    mask_train = np.asarray(mask_train, dtype=np.float32)
    Data_test = np.asarray(Data_test, dtype=np.float32)
    mask_test = np.asarray(mask_test, dtype=np.float32)

    n_test = Data_test.shape[0]
    OBS_DIM = Data_test.shape[1]

    # allocate result arrays once (same behavior as your original)
    information_curve_RAND = np.zeros((Repeat, n_test, (OBS_DIM - 1) + 1), dtype=np.float32)
    information_curve_SING = np.zeros((Repeat, n_test, (OBS_DIM - 1) + 1), dtype=np.float32)
    information_curve_CHAI = np.zeros((Repeat, n_test, (OBS_DIM - 1) + 1), dtype=np.float32)

    action_SING = np.zeros((Repeat, n_test, OBS_DIM - 1), dtype=np.int32)
    action_CHAI = np.zeros((Repeat, n_test, OBS_DIM - 1), dtype=np.int32)

    R_hist_SING = np.zeros((Repeat, OBS_DIM - 1, n_test, OBS_DIM - 1), dtype=np.float32)
    R_hist_CHAI = np.zeros((Repeat, OBS_DIM - 1, n_test, OBS_DIM - 1), dtype=np.float32)

    im_SING = np.zeros((Repeat, OBS_DIM - 1, M, n_test, OBS_DIM), dtype=np.float32)
    im_CHAI = np.zeros((Repeat, OBS_DIM - 1, M, n_test, OBS_DIM), dtype=np.float32)

    for r in range(Repeat):
        # TF2 replacement of tf.reset_default_graph()
        tf.keras.backend.clear_session()
        np.random.seed(42 + r)
        random.seed(42 + r)

        # train partial VAE (TF2)
        vae = train_p_vae(
            Data_train=Data_train,
            mask_train=mask_train,
            epochs=epochs,
            latent_dim=latent_dim,
            batch_size=batch_size,
            p=p,
            K=K,
            iteration=10,  # 你原代码传入的 iteration
            obs_distrib=obs_distrib,
            obs_std=obs_std,
            encoder_weights_path=encoder_weights_path,
            decoder_weights_path=decoder_weights_path,
            # 若想尽量复刻旧代码（Gaussian但decoder sigmoid），可显式设 "sigmoid"
            # decoder_final_activation="sigmoid",
        )

        # ---------- strategy loop ----------
        for strategy in range(3):
            x = Data_test.copy().reshape(n_test, OBS_DIM)

            # mask: tracks observed/unobserved during active learning
            mask = np.zeros((n_test, OBS_DIM), dtype=np.float32)
            mask[:, -1] = 0.0  # never observe target

            # mask2: tracks which features have been selected (for SING bookkeeping)
            mask2 = np.zeros((n_test, OBS_DIM), dtype=np.float32)

            # initial evaluation (no observation)
            neg_loss, _ = vae.predictive_loss(x, mask, eval, M)
            if strategy == 0:
                information_curve_RAND[r, :, 0] = neg_loss
            elif strategy == 1:
                information_curve_SING[r, :, 0] = neg_loss
            else:
                information_curve_CHAI[r, :, 0] = neg_loss

            # -------- strategy 0: RAND --------
            if strategy == 0:
                i_optimal = np.tile(np.arange(OBS_DIM - 1), (n_test, 1))
                for row in i_optimal:
                    random.shuffle(row.tolist())

                for t in range(OBS_DIM - 1):
                    print(f"Repeat={r} Strategy=RAND Step={t}")
                    io = np.eye(OBS_DIM, dtype=np.float32)[i_optimal[:, t]]
                    mask = mask + io
                    neg_loss, _ = vae.predictive_loss(x, mask, eval, M)
                    information_curve_RAND[r, :, t + 1] = neg_loss

            # -------- strategy 1: SING --------
            elif strategy == 1:
                M_eval = M  # 曲线评估用大 M
                M_reward = 5  # reward 用小 M（你也可以外部传参）
                subset_size = 64  # 只用子集估计全局 i_star（None=全量）

                # 用于估计 R.mean 的子集（每个 step 都固定同一批也行）
                if subset_size is None or subset_size >= n_test:
                    subset_idx = np.arange(n_test, dtype=np.int32)
                else:
                    rng = np.random.default_rng(1000 + r)  # 固定种子，保证可复现
                    subset_idx = rng.choice(n_test, size=subset_size, replace=False).astype(np.int32)

                for t in range(OBS_DIM - 1):
                    print(f"Repeat={r} Strategy=SING Step={t}")
                    R = -1e4 * np.ones((n_test, OBS_DIM - 1), dtype=np.float32)

                    # reward 用小 M
                    im_0 = completion(x, mask * 0.0, M_reward, vae)
                    im   = completion(x, mask,       M_reward, vae)
                    # 你如果还想存 im_SING 用于分析，可以只存前 M_reward，避免内存爆
                    im_SING[r, t, :M_reward, :, :] = im

                    # 只在 subset 上算 reward（因为最后只用 R.mean(axis=0)）
                    for u in range(OBS_DIM - 1):
                        # 这个 feature 还没被选过的样本
                        locs = np.where(mask2[:, u] == 0)[0]
                        if locs.size == 0:
                            continue

                        # 只取 subset
                        locs = locs[np.isin(locs, subset_idx)]
                        if locs.size == 0:
                            continue

                        for l in locs:
                            if estimation_method == 0:
                                R[l, u] = R_lindley_chain(u, x, mask, M_reward, vae, im_0, l)
                            else:
                                R[l, u] = R_lindley_chain(u, x, mask, M_reward, vae, im, l)

                    R_hist_SING[r, t, :, :] = R

                    # 注意：mean 只对 subset 做，避免大面积 -1e4 污染均值
                    R_mean = R[subset_idx, :].mean(axis=0)
                    i_star = int(np.argmax(R_mean))
                    i_optimal = np.full((n_test,), i_star, dtype=np.int32)

                    io = np.eye(OBS_DIM, dtype=np.float32)[i_optimal]
                    action_SING[r, :, t] = i_optimal

                    mask  = np.clip(mask  + io * mask_test, 0.0, 1.0)
                    mask2 = np.clip(mask2 + io,             0.0, 1.0)

                    # 曲线评估仍用大 M
                    neg_loss, _ = vae.predictive_loss(x, mask, eval, M_eval)
                    information_curve_SING[r, :, t + 1] = neg_loss

              # -------- strategy 2: CHAI / EDDI (chain rule approx) --------
            else:
                M_eval = M      # 画曲线用大 M（稳定）
                M_reward = 5    # 选特征用小 M（快很多）

                for t in range(OBS_DIM - 1):
                    print(f"Repeat={r} Strategy=CHAI Step={t}")
                    R = -1e4 * np.ones((n_test, OBS_DIM - 1), dtype=np.float32)

                    # reward 用小 M 的 completion
                    im = completion(x, mask, M_reward, vae)

                    # 如果你一定要存 im_CHAI（会很占内存），也只存前 M_reward
                    im_CHAI[r, t, :M_reward, :, :] = im

                    for u in range(OBS_DIM - 1):
                        locs = np.where(mask[:, u] == 0)[0]
                        if locs.size == 0:
                            continue

                        for l in locs:
                            R[l, u] = R_lindley_chain(u, x, mask, M_reward, vae, im, l)

                    R_hist_CHAI[r, t, :, :] = R

                    # personalized: one best feature per test instance
                    i_optimal = R.argmax(axis=1).astype(np.int32)
                    io = np.eye(OBS_DIM, dtype=np.float32)[i_optimal]

                    action_CHAI[r, :, t] = i_optimal
                    mask  = np.clip(mask  + io, 0.0, 1.0)
                    mask2 = np.clip(mask2 + io, 0.0, 1.0)

                    # 曲线评估仍用大 M
                    neg_loss, _ = vae.predictive_loss(x, mask, eval, M_eval)
                    information_curve_CHAI[r, :, t + 1] = neg_loss

    # ---- Save results (same filenames) ----
    os.makedirs(output_dir, exist_ok=True)

    np.savez(os.path.join(output_dir, "UCI_information_curve_RAND.npz"), information_curve=information_curve_RAND)
    np.savez(os.path.join(output_dir, "UCI_information_curve_SING.npz"), information_curve=information_curve_SING)
    np.savez(os.path.join(output_dir, "UCI_information_curve_CHAI.npz"), information_curve=information_curve_CHAI)

    np.savez(os.path.join(output_dir, "UCI_action_SING.npz"), action=action_SING)
    np.savez(os.path.join(output_dir, "UCI_action_CHAI.npz"), action=action_CHAI)

    np.savez(os.path.join(output_dir, "UCI_R_hist_SING.npz"), R_hist=R_hist_SING)
    np.savez(os.path.join(output_dir, "UCI_R_hist_CHAI.npz"), R_hist=R_hist_CHAI)

    np.savez(os.path.join(output_dir, "UCI_im_SING.npz"), im=im_SING)
    np.savez(os.path.join(output_dir, "UCI_im_CHAI.npz"), im=im_CHAI)

    return None



def train_p_vae(
    Data_train,
    mask_train,
    epochs,
    latent_dim,
    batch_size,
    p,
    K,
    iteration,
    obs_distrib="Gaussian",
    obs_std=0.1 * np.sqrt(2),
    encoder_weights_path=None,
    decoder_weights_path=None,
    decoder_final_activation=None,  # None for Gaussian mean; "sigmoid" if你要复刻旧代码
):
    """
    TF2/Keras training loop for partial VAE.

    Data_train: [N,D]
    mask_train: [N,D] 1 observed 0 missing
    p: dropout rate (your original code uses per-entry random in [0,p])
    iteration: -1 full epoch else fixed number of minibatches per epoch
    """

    Data_train = np.asarray(Data_train, dtype=np.float32)
    mask_train = np.asarray(mask_train, dtype=np.float32)

    obs_dim = Data_train.shape[1]
    n_train = Data_train.shape[0]
    idx_all = np.arange(n_train)

    # ---- build model (TF2) ----
    if decoder_final_activation is None:
        # 旧代码 fc_uci_decoder 是 sigmoid 输出；但你设 obs_distrib="Gaussian"
        # 如果你想“尽量复刻旧结果”，可以把下面改成 "sigmoid"
        decoder_final_activation = None if obs_distrib.lower() == "gaussian" else "sigmoid"

    decoder = FCUCIDecoder(obs_dim=obs_dim, final_activation=decoder_final_activation)

    vae = PNPlusVAE(
        obs_dim=obs_dim,
        decoder=decoder,
        K=K,
        latent_dim=latent_dim,
        obs_distrib=obs_distrib,
        obs_std=obs_std,
        learning_rate=1e-3,
        M=5,
    )

    # 如果你希望 warm-start / load pretrain（可选）
    if encoder_weights_path and os.path.exists(encoder_weights_path):
        vae.load_encoder(encoder_weights_path)
    if decoder_weights_path and os.path.exists(decoder_weights_path):
        vae.load_decoder(decoder_weights_path)

    # number of iterations per epoch
    if iteration == -1:
        n_it = int(np.ceil(n_train / float(batch_size)))
    else:
        n_it = int(iteration)

    for epoch in range(epochs):
        training_loss_full = 0.0

        for it in range(n_it):
            if iteration == -1:
                start = it * batch_size
                end = min(start + batch_size, n_train)
                batch_indices = idx_all[start:end]
            else:
                batch_indices = sample(range(n_train), batch_size)

            x = Data_train[batch_indices, :]
            mask_train_batch = mask_train[batch_indices, :]
            B = x.shape[0]

            # ---- your dropout mask logic (adapted to variable batch size) ----
            DROPOUT_TRAIN = np.minimum(np.random.rand(B, obs_dim), p)

            while True:
                mask_drop = bernoulli.rvs(1 - DROPOUT_TRAIN).astype(np.float32)
                if np.sum(mask_drop > 0) > 0:
                    break

            mask_eff = mask_drop * mask_train_batch  # combine data missingness + dropout missingness

            _ = vae.update(x, mask_eff)
            loss_full, _, _ = vae.full_batch_loss(x, mask_eff)
            training_loss_full += loss_full

        training_loss_full /= n_it
        print(f"Epoch: {epoch}\tnegative training ELBO per observed feature: {training_loss_full:.2f}")

    # ---- save weights (TF2) ----
    if decoder_weights_path:
        vae.save_decoder(decoder_weights_path)
    if encoder_weights_path:
        vae.save_encoder(encoder_weights_path)

    return vae

In [8]:
Data_df = pd.concat([X_df, y_df], axis=1).dropna().reset_index(drop=True)

Data = Data_df.to_numpy(dtype=np.float32)

# ---- 按你贴的逻辑：先按列 min-max 到 [0,1]，再映射到 [min_Data, max_Data] ----
Data_std = (Data - Data.min(axis=0, keepdims=True)) / (Data.max(axis=0, keepdims=True) - Data.min(axis=0, keepdims=True) + 1e-8)

min_Data, max_Data = 0.0, 1.0   # 论文代码里通常就是 0/1，这样 Data 就是 Data_std
Data = Data_std * (max_Data - min_Data) + min_Data

Mask = np.ones_like(Data, dtype=np.float32)   # wine 完全观测

Data_train, Data_test, mask_train, mask_test = train_test_split(
    Data, Mask, test_size=0.1, random_state=rs, shuffle=True
)

print("Data_train:", Data_train.shape, "Data_test:", Data_test.shape, "D=", Data.shape[1])
print("Target col name:", Data_df.columns[-1])

Data_train: (5847, 12) Data_test: (650, 12) D= 12
Target col name: quality


In [9]:
rs = 42
epochs = 300
latent_dim = 10
batch_size = 100
p = 0.7
K = 20
M = 5
Repeat = 3

eval_metric = "rmse"          # 你画 Figure 9 用这个
estimation_method = 0         # 0=用 im_0; 1=用 im（按你代码）
output_dir = "./wine_active_learning_out"
os.makedirs(output_dir, exist_ok=True)

# 如果你想 warm-start（可选）
encoder_weights_path = None
decoder_weights_path = None

# -----------------------
# 2) 运行 active learning（会保存 npz 到 output_dir）
# -----------------------
_ = p_vae_active_learning(
    Data_train, mask_train,
    Data_test, mask_test,
    epochs, latent_dim, batch_size,
    p, K, M, eval_metric, Repeat,
    estimation_method=estimation_method,
    encoder_weights_path=encoder_weights_path,
    decoder_weights_path=decoder_weights_path,
    output_dir=output_dir,
    obs_distrib="Gaussian",
    obs_std=0.1 * np.sqrt(2),
)

print("Done. Saved results to:", output_dir)

# -----------------------
# 3) 直接画图（复刻你原 repo 的逻辑）
# -----------------------
IC_RAND = np.load(output_dir + "/UCI_information_curve_RAND.npz")["information_curve"]
IC_SING = np.load(output_dir + "/UCI_information_curve_SING.npz")["information_curve"]
IC_CHAI = np.load(output_dir + "/UCI_information_curve_CHAI.npz")["information_curve"]

fig, ax1 = plt.subplots()

ax1.plot(np.sqrt((IC_RAND[:, :, 0:].mean(axis=0)).mean(axis=0)), "gs", linestyle="-.", label="PNP+RAND")
ax1.errorbar(
    np.arange(IC_RAND.shape[2]),
    np.sqrt((IC_RAND[:, :, 0:].mean(axis=0)).mean(axis=0)),
    yerr=np.sqrt((IC_RAND[:, :, 0:]).mean(axis=1)).std(axis=0) / np.sqrt(IC_SING.shape[0]),
    ecolor="g",
    fmt="gs",
)

ax1.plot(np.sqrt((IC_SING[:, :, 0:].mean(axis=0)).mean(axis=0)), "ms", linestyle="-.", label="PNP+SING")
ax1.errorbar(
    np.arange(IC_SING.shape[2]),
    np.sqrt((IC_SING[:, :, 0:].mean(axis=0)).mean(axis=0)),
    yerr=np.sqrt((IC_SING[:, :, 0:]).mean(axis=1)).std(axis=0) / np.sqrt(IC_SING.shape[0]),
    ecolor="m",
    fmt="ms",
)

ax1.plot(np.sqrt((IC_CHAI[:, :, 0:].mean(axis=0)).mean(axis=0)), "ks", linestyle="-.", label="PNP+EDDI")
ax1.errorbar(
    np.arange(IC_CHAI.shape[2]),
    np.sqrt((IC_CHAI[:, :, 0:].mean(axis=0)).mean(axis=0)),
    yerr=np.sqrt((IC_CHAI[:, :, 0:]).mean(axis=1)).std(axis=0) / np.sqrt(IC_SING.shape[0]),
    ecolor="k",
    fmt="ks",
)

plt.xlabel("Steps", fontsize=18)
plt.ylabel("avg. test RMSE", fontsize=18)
plt.xticks(fontsize=18)
plt.yticks(fontsize=18)
ax1.legend(
    bbox_to_anchor=(0.0, 1.02, 1.0, 0.102),
    mode="expand",
    loc=3,
    ncol=1,
    borderaxespad=0.0,
    prop={"size": 20},
    frameon=False,
)

plt.show()
plt.savefig(output_dir + "/PNP_all_IC_curves.png", format="png", dpi=200, bbox_inches="tight")
print("Saved figure:", output_dir + "/PNP_all_IC_curves.png")

Epoch: 0	negative training ELBO per observed feature: -0.33
Epoch: 1	negative training ELBO per observed feature: -1.07
Epoch: 2	negative training ELBO per observed feature: -1.20
Epoch: 3	negative training ELBO per observed feature: -1.31
Epoch: 4	negative training ELBO per observed feature: -1.35
Epoch: 5	negative training ELBO per observed feature: -1.41
Epoch: 6	negative training ELBO per observed feature: -1.45
Epoch: 7	negative training ELBO per observed feature: -1.47
Epoch: 8	negative training ELBO per observed feature: -1.48
Epoch: 9	negative training ELBO per observed feature: -1.52
Epoch: 10	negative training ELBO per observed feature: -1.51
Epoch: 11	negative training ELBO per observed feature: -1.52
Epoch: 12	negative training ELBO per observed feature: -1.52
Epoch: 13	negative training ELBO per observed feature: -1.53
Epoch: 14	negative training ELBO per observed feature: -1.57
Epoch: 15	negative training ELBO per observed feature: -1.56
Epoch: 16	negative training ELBO p

KeyboardInterrupt: 